# --------------------------------------Introduction--------------------------------------
## XGBoost (Extreme Gradient Boosting):
### What it is:
* `XGBoost(Extreme Gradient Boosting)` is a machine learning algorithm — specifically, it's an implementation of gradient boosting, and it's one of the most widely used models for `tabular` data (like Melbourne housing dataset) in both real-world use and Kaggle competitions. `Builds decision trees sequentially (one at a time)`, unlike Random Forest which `builds trees independently/in parallel`.

### How it works:

1. Start with a `baseline prediction (e.g. average price)`
2. Tree 1 is trained to `predict the error (baseline vs actual)`
3. Add Tree 1's correction to the `baseline → new prediction, new (smaller) error`
4. Tree 2 is trained to predict that `remaining error → add it → error shrinks again`
5. Repeat for many trees, each one `correcting what's still wrong`
6. Final prediction = `baseline + sum of all trees corrections (sum, not average)`

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

data=pd.read_csv("melb_data.csv")

cols_to_use = ['Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']

X=data[cols_to_use]

y=data.Price

train_X,valid_X,train_y,valid_y=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=0)

In [3]:
from xgboost import XGBRegressor
def calculate_error(model,valid_X,valid_y):
    prediction=model.predict(valid_X)
    error=mean_absolute_error(valid_y,prediction)
    return error

In [4]:
from sklearn.metrics import mean_absolute_error
model=XGBRegressor()
model.fit(train_X,train_y)
print("Mean Absolute Error: ",calculate_error(model,valid_X,valid_y))

Mean Absolute Error:  241657.32839538844


# Parameter Tuning:
### XGBoost has a few parameters that can dramatically affect accuracy and training speed.
#### 1. n_estimators:

`n_estimators` specifies how many times to go through the modeling cycle described above. It is equal to the number of models that we include in the ensemble.

* Too low a value causes underfitting, which leads to inaccurate predictions on both training data and test data.
* Too high a value causes overfitting, which causes accurate predictions on training data, but inaccurate predictions on test data (which is what we care about).

In [6]:
model=XGBRegressor(n_estimators=500)
model.fit(train_X,train_y)
print("Mean Absolute Error: ",calculate_error(model,valid_X,valid_y))

Mean Absolute Error:  250963.21331059


#### 2.early_stopping_rounds:
* `early_stopping_round` offers a way to automatically find the ideal value for `n_estimators`. Early stopping causes the model to stop iterating when the validation score stops improving. It's smart to set a high value for `n_estimators` and then use `early_stopping_rounds` to find the optimal time to stop iterating.
* Setting early_stopping_rounds=5 is a reasonable choice.
* When using `early_stopping_rounds`, we also need to set aside some data for calculating the validation scores - this is done by setting the `eval_set` parameter.